# PAYBACK Lightweight Assistant — Performance Report
Load test against the live Cloud Run deployment `https://payback-assistant-506488371374.europe-west3.run.app` (1 vCPU / 512 MiB per instance, scale-to-zero, max 3 instances, concurrency 80).

In [1]:
import json, statistics, time, requests
from concurrent.futures import ThreadPoolExecutor
BASE = 'https://payback-assistant-506488371374.europe-west3.run.app'
QUERIES = ['günstige Windeln', 'Suche Shampoo bei dm', 'pasta tomaten parmesan',
           'Schokolade und Chips', 'Mineralwasser 6er']  # rules path = steady-state hot path
def run_load(n_requests, concurrency):
    latencies, errors = [], 0
    session = requests.Session()
    def one(i):
        nonlocal errors
        t0 = time.perf_counter()
        try:
            r = session.post(f'{BASE}/assist', json={'query': QUERIES[i % len(QUERIES)]}, timeout=30)
            r.raise_for_status()
            latencies.append((time.perf_counter() - t0) * 1000)
        except Exception:
            errors += 1
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        list(pool.map(one, range(n_requests)))
    wall = time.perf_counter() - t0
    latencies.sort()
    p = lambda q: latencies[min(len(latencies) - 1, int(q * len(latencies)))]
    return {'requests': n_requests, 'concurrency': concurrency, 'errors': errors,
            'wall_s': round(wall, 2), 'rps': round(len(latencies) / wall, 1),
            'p50_ms': round(p(.5), 1), 'p95_ms': round(p(.95), 1), 'p99_ms': round(p(.99), 1)}
requests.get(f'{BASE}/health', timeout=30).json()  # warm the instance

{'status': 'ok',
 'version': '1.0.0',
 'products': 97,
 'llm': 'gemini-2.5-flash-lite'}

## Scaling proof: throughput grows with concurrency, tail latency stays bounded

In [2]:
results = [run_load(150, c) for c in (5, 20, 40)]
for r in results:
    print(r)

{'requests': 150, 'concurrency': 5, 'errors': 0, 'wall_s': 4.69, 'rps': 32.0, 'p50_ms': 136.7, 'p95_ms': 152.3, 'p99_ms': 607.7}
{'requests': 150, 'concurrency': 20, 'errors': 0, 'wall_s': 2.5, 'rps': 60.1, 'p50_ms': 136.6, 'p95_ms': 1521.0, 'p99_ms': 1533.8}
{'requests': 150, 'concurrency': 40, 'errors': 0, 'wall_s': 3.43, 'rps': 43.8, 'p50_ms': 153.2, 'p95_ms': 2792.5, 'p99_ms': 2840.9}


## LLM model tiers: server-side latency per Gemini model
Same paraphrase queries, `llm_mode: always`, measured via the response's own `latency_ms` (server-side handler time incl. the Vertex AI call — network round-trip to the client excluded).

**LLM used** counts responses whose `engine` shows the model actually answered; the remainder hit the service's 6 s LLM timeout (plus one retry) and fell back to the deterministic rules path — the ~12 s latencies of slow tiers are the two exhausted timeouts, not usable inference. `always` means *always attempt* the LLM; the timeout guard means the API still answers when a tier can't keep up.

In [3]:
MODELS = ['gemini-2.5-flash-lite', 'gemini-2.5-flash', 'gemini-3.1-flash-lite', 'gemini-3.5-flash']
PARAPHRASES = ['etwas gegen wunden Po beim Kleinkind', 'pancakes for the kids',
               'was Schönes zum Naschen', 'stuff for taco night', 'spiegelei fürs frühstück',
               'something for a rainy sunday afternoon']
print(f"{'model':26} {'p50 ms':>8} {'p95 ms':>8} {'mean ms':>8} {'LLM used':>9}  verdict")
for m in MODELS:
    lats, used = [], 0
    for q in PARAPHRASES:
        r = requests.post(f'{BASE}/assist', json={'query': q, 'llm_mode': 'always',
                          'model': m, 'user_id': f'tier-{m}'}, timeout=60).json()
        lats.append(r['latency_ms'])
        used += m in r['engine']  # engine names the model only when it answered
    lats.sort()
    n = len(PARAPHRASES)
    verdict = 'ok' if used == n else (f'{n - used} timeout fallback(s) to rules')
    print(f'{m:26} {lats[len(lats)//2]:8.0f} {lats[-1]:8.0f} {statistics.mean(lats):8.0f} '
          f'{used}/{n:>3}  {verdict}')

model                        p50 ms   p95 ms  mean ms  LLM used  verdict


gemini-2.5-flash-lite           573      617      546 6/  6  ok


gemini-2.5-flash               4069    12361     6263 4/  6  2 timeout fallback(s) to rules


gemini-3.1-flash-lite           977     1286      917 6/  6  ok


gemini-3.5-flash               5401    12377     7426 4/  6  2 timeout fallback(s) to rules


## Cost per 1000 requests (measured, Cloud Run request-based billing)
vCPU $0.000024/s + memory $0.0000025/GiB·s billed only while serving, plus $0.40 per million requests. LLM-escalated requests add Gemini 2.5 Flash-Lite on Vertex AI (~$0.10/M input + $0.40/M output tokens; ~700 in / 60 out per call).

In [4]:
best = max(results, key=lambda r: r['rps'])
per_req_s = 1 / best['rps']
infra = (0.000024 + 0.0000025 * 0.5) * per_req_s + 0.40 / 1_000_000
llm_call = 700 / 1e6 * 0.10 + 60 / 1e6 * 0.40   # per escalated call
for esc in (0.0, 0.10, 0.25):
    total = (infra + esc * llm_call) * 1000
    print(f'cost per 1000 requests at {esc:.0%} LLM escalation: ${total:.4f}')

cost per 1000 requests at 0% LLM escalation: $0.0008
cost per 1000 requests at 10% LLM escalation: $0.0102
cost per 1000 requests at 25% LLM escalation: $0.0243


## How prompt/inference time was optimized
Latency was minimized by **removing inference from the hot path rather than optimizing it**: deterministic lexicon rules + a concept-grouped BM25 index answer every query whose vocabulary is known in ~1–2 ms server-side, so the p50 above is pure network + HTTP. The LLM (Gemini 2.5 Flash-Lite — the smallest, fastest tier) is called **only** when the query contains tokens unknown to the index, which empirically is a small fraction of retail traffic; every miss that recurs can be promoted into the synonym lexicon, shrinking the escalation rate over time. The prompt itself is optimized for speed: a single compact instruction block, few-shot examples instead of long specifications, `responseMimeType: application/json` to eliminate parsing retries, temperature 0, and a hard 6 s timeout with one retry falling back to the rules answer — the API never blocks on the LLM. The user-context profile is injected as one short line (top-5 categories, 30% weight) rather than raw history, keeping input tokens ~700 per escalated call. Serving-side, the index is built once at startup from a catalog baked into the image (no cold-start I/O), the service is stateless, and Cloud Run fans out to 3 instances × concurrency 80.